In [92]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph,START,END,MessagesState
from dotenv import load_dotenv
from typing import TypedDict
from pprint import pprint

In [93]:
class BatsmanState(TypedDict):
    runs:int
    fours:int
    sixes:int
    balls:int
    sr:float
    bpb:float
    bp:float
    summary:str

In [94]:
load_dotenv()

True

In [95]:
model=ChatGoogleGenerativeAI(  model="gemini-3.5-flash",temperature=0)

In [96]:
def calculate_sr(state):
    sr = (state["runs"] / state["balls"]) * 100
    return {"sr": sr}

In [97]:
def bpb(state):
    boundary = state["fours"] + state["sixes"]
    bpb = boundary / state["balls"]
    return {"bpb": bpb}

In [98]:
def bp(state):
    bruns = state["fours"] * 4 + state["sixes"] * 6
    bp_value = (bruns / state["runs"]) * 100
    return {"bp": bp_value}

In [99]:
def Summary(state:BatsmanState):
    summary=f"""
Strike rate -{state['sr']}\n
balls per boundarry -{state['bpb']}\n
boundary percentage-{state['bp']}
"""
    state['summary']=summary

    return state

In [100]:
graph=StateGraph(BatsmanState)

In [101]:
graph.add_node('calculate_sr',calculate_sr)
graph.add_node('bpb',bpb)
graph.add_node('bp',bp)
graph.add_node('summary',Summary)


graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'bpb')
graph.add_edge(START,'bp')

graph.add_edge('calculate_sr','summary')
graph.add_edge('bpb','summary')
graph.add_edge('bp','summary')

graph.add_edge('summary',END)


workflow=graph.compile()



In [102]:
initial_state={
    'runs':100,
    'balls':50,
    'sixes':5,
    'fours':10
}

final_state=workflow.invoke(initial_state)
print(final_state)

{'runs': 100, 'fours': 10, 'sixes': 5, 'balls': 50, 'sr': 200.0, 'bpb': 0.3, 'bp': 70.0, 'summary': '\nStrike rate -200.0\n\nballs per boundarry -0.3\n\nboundary percentage-70.0\n'}
